# 📘 Домашнє завдання 25. Введення в Agentic AI

Виконав: **Bohdan Pinchuk**

Link: https://github.com/BogdanPinchuk/DataScience-PBY_HW25


## Мета

Створити агента на базі LangGraph, який відповідає на запитання про історію машинного навчання, використовуючи RAG (Retrieval-Augmented Generation).

### Джерело знань

Стаття:

https://arxiv.org/pdf/2408.01747.pdf

**Classical Machine Learning: Seventy Years of Algorithmic Learning Evolution**

---

## Що повинен робити агент?

Користувач ставить запитання:

```text
What are the main milestones in machine learning?
```

або

```text
Which algorithms were influential in the 1990s?
```

Агент:

1. Отримує питання.
2. Шукає релевантні фрагменти статті.
3. Передає знайдений контекст до LLM.
4. Генерує відповідь на основі контексту.

---

## Архітектура

```text
START
  ↓
Retrieve Documents
  ↓
Generate Answer
  ↓
END
```

---

## Використати

### LLM

```python
Qwen/Qwen2.5-1.5B-Instruct
```

### Embeddings

```python
sentence-transformers/all-MiniLM-L6-v2
```

---

## Встановлення

```bash
pip install \
langgraph \
langchain \
langchain-community \
sentence-transformers \
faiss-cpu \
pypdf \
transformers \
torch
```

---

# Частина 1. State

```python
from typing import TypedDict

class RAGState(TypedDict):
    """
    Стан агента.
    """

    question: str

    context: str

    answer: str
```

---

# Частина 2. Завантаження PDF

```python
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    "2408.01747.pdf"
)

documents = loader.load()
```

---

# Частина 3. Розбиття на чанки

Заповніть пропуски.

```python
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=__________,
    chunk_overlap=__________
)

chunks = splitter.split_documents(
    documents
)
```

---

# Частина 4. Створення ембедінгів

```python
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
```

---

# Частина 5. Створення векторної бази

Заповніть один пропуск.

```python
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    __________
)
```

---

# Частина 6. Retriever

```python
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3
    }
)
```

---

# Частина 7. Ініціалізація Qwen

```python
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_new_tokens=300
)
```

---

# Частина 8. Функція виклику LLM

Заповніть один пропуск.

```python
def ask_llm(prompt: str):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = llm(
        __________,
        return_full_text=False
    )

    return response[0]["generated_text"]
```

---

# Частина 9. Node 1 — Retrieval

Заповніть пропуски.

```python
def retrieve_documents(state: RAGState):
    """
    Пошук релевантних фрагментів статті.
    """

    docs = retriever.invoke(
        __________
    )

    context = "\n\n".join(
        [
            d.page_content
            for d in docs
        ]
    )

    return {
        "context": context
    }
```

---

# Частина 10. Node 2 — Answer Generation

Заповніть один пропуск.

```python
def generate_answer(state: RAGState):
    """
    Генерує відповідь на основі
    знайдених фрагментів статті.
    """

    prompt = f"""
Answer the question using ONLY the context.

Question:
{state['question']}

Context:
{state['context']}
"""

    answer = __________(prompt)

    return {
        "answer": answer
    }
```

---

# Частина 11. Побудова графа

Заповніть пропуски.

```python
from langgraph.graph import (
    StateGraph,
    START,
    END
)

builder = StateGraph(RAGState)

builder.add_node(
    "retrieve",
    retrieve_documents
)

builder.add_node(
    "answer",
    generate_answer
)

builder.add_edge(
    START,
    __________
)

builder.add_edge(
    __________,
    __________
)

builder.add_edge(
    __________,
    END
)

graph = builder.compile()
```

---

# Частина 12. Запуск

```python
result = graph.invoke(
    {
        "question":
        "What are the major milestones in machine learning?"
    }
)

print(result["answer"])
```

### Очікувана архітектура

```text
START
  ↓
Rewrite Question
  ↓
Retrieve Documents
  ↓
Generate Answer
  ↓
END
```
